In [77]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm import tqdm

In [78]:
dataset = load_dataset("openai/gsm8k", "main")

In [79]:
count = 0
while count < 5:
    print(f"question : {dataset['train']['question'][count]}\n answer : {dataset['train']['answer'][count]}\n")
    count += 1

question : Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
 answer : Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

question : Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
 answer : Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

question : Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
 answer : In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = 

In [80]:
print(f"train dataset length: {len(dataset["train"])}\ntest dataset length: {len(dataset['test'])}")

train dataset length: 7473
test dataset length: 1319


In [81]:
train_questions =  dataset["train"]["question"]
train_answers = dataset["train"]["answer"]

# Combine questions and answers into pairs
test_pairs = list(zip(dataset["test"]["question"], dataset["test"]["answer"]))

# Split pairs into validation and test sets
test_pairs, val_pairs = train_test_split(test_pairs, test_size=0.5, random_state=42)

# Unzip pairs back into separate lists
test_questions, test_answers = zip(*test_pairs)
val_questions, val_answers = zip(*val_pairs)

In [82]:
#Tokenize words of train, test and validation datasets

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Use the same max_length for both questions and answers to ensure matching sequence lengths
max_length = 64

train_encodings_question = tokenizer(train_questions, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")
val_encodings_question = tokenizer(val_questions, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")
test_encodings_question = tokenizer(test_questions, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")

train_encodings_answer = tokenizer(train_answers, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")
val_encodings_answer = tokenizer(val_answers, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")
test_encodings_answer = tokenizer(test_answers, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")

In [83]:
class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.labels["input_ids"][idx],
        }


train_dataset = CustomDataset(train_encodings_question, train_encodings_answer)
val_dataset = CustomDataset(val_encodings_question, val_encodings_answer)
test_dataset = CustomDataset(test_encodings_question, test_encodings_answer)

In [84]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [85]:
print(train_loader)

In [86]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [87]:
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to(device)

In [97]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # model.eval()
    # total_loss = 0
    # with torch.no_grad():
    #     for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}"):
    #         input_ids = batch['input_ids'].to(device)
    #         attention_mask = batch['attention_mask'].to(device)
    #         labels = batch['labels'].to(device)

    #         outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    #         total_loss += outputs.loss.item()

    # avg_val_loss = total_loss / len(val_loader)
    # print(f"Epoch {epoch + 1}, Validation Loss: {avg_val_loss:.4f}")

Training Epoch 3: 100%|██████████| 935/935 [11:33<00:00,  1.35it/s]


In [99]:
def generate_answer(question):
    inputs = tokenizer(question, return_tensors="pt").to("mps")
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

question = "Natalia sold clips to 80 friends in April and half as many in May. How many clips did she sell in total?"
print(generate_answer(question))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Natalia sold clips to 80 friends in April and half as many in May. How many clips did she sell in total?>>== =


In [ ]:
# Download the raw JSON files
!wget https://raw.githubusercontent.com/openai/grade-school-math/refs/heads/master/grade_school_math/data/train.jsonl
!wget https://raw.githubusercontent.com/openai/grade-school-math/refs/heads/master/grade_school_math/data/test.jsonl

# Convert to HF dataset format
import json
from datasets import Dataset, DatasetDict

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load the data
train_data = load_jsonl('/content/train.jsonl')
test_data = load_jsonl('/content/test.jsonl')

# Create dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("Dataset created successfully!")
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

Dataset created successfully!
Train set size: 7473
Test set size: 1319


In [108]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})